# Test.csv 3-Model EN->KO Translation Eval

- Fixed model set (only 3):
  1. `alwaysgood/qwen3-it`
  2. `alwaysgood/qwen35-it`
  3. `alwaysgood/gemma4-it`
- COMET is mandatory (`XCOMET-XXL -> XCOMET-XL` fallback only).

## One-Click Checklist
1. (First run only) Run dependency cell (`%pip ...`).
2. Click **Kernel -> Restart & Run All**.
3. Check outputs in `OUTPUT_DIR`:
   - `translations.csv`
   - `translations_by_model_wide.csv`
   - `row_level_scores.csv`
   - `model_summary.csv`

## Quick OOM Tuning
- If model generation OOM: lower `BATCH_SIZE_QWEN` / `BATCH_SIZE_GEMMA`.
- If COMET OOM: lower `COMET_BATCH_SIZE`.
- `MAX_NEW_TOKENS` means **generated tokens only** (prompt length excluded).



In [ ]:
# Runtime bootstrap: install packages into the active notebook kernel env
# (vast.ai / Lightning AI / Colab compatible)
import subprocess
import sys


def pip_install(args, optional=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(args)
    print('running:', ' '.join(cmd))
    try:
        subprocess.check_call(cmd)
        return True
    except Exception as e:
        if optional:
            print(f'[WARN] optional install failed: {args} -> {type(e).__name__}: {e}')
            return False
        raise


# Core stack (same style as eval120_stage2_sequential.ipynb)
pip_install([
    'unsloth',
    'unsloth-zoo',
    'datasets',
    'trl',
    'hydra-core',
    'omegaconf',
    'sentencepiece',
    'numpy',
])

# Eval / utility stack
pip_install([
    'lm-eval',
    'mergekit',
    'pandas',
    'tabulate',
    'matplotlib',
    'seaborn',
    'tqdm',
    'sacrebleu',
    'rouge-score',
    'bert-score',
    'unbabel-comet==2.2.7',
    'pyyaml>=6.0.2',
])

# Optional speed stack
XFORMERS_INSTALL_OK = pip_install(['xformers'], optional=True)
print(f'xformers_install_ok={XFORMERS_INSTALL_OK}')

# Pin transformers exactly as requested
pip_install(['transformers==5.5.4', 'trl>=0.15.0', '--no-deps'])

print('bootstrap install done')
print('Restart kernel, then Run All.')



In [ ]:
import gc
import os
import re
import shutil
import warnings
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoTokenizer

warnings.filterwarnings('ignore')

HAS_UNSLOTH = importlib.util.find_spec('unsloth') is not None
if not HAS_UNSLOTH:
    raise RuntimeError('This notebook requires unsloth for inference, but unsloth is not installed.')

if transformers.__version__ != '5.5.4':
    raise RuntimeError(f"Expected transformers==5.5.4, got {transformers.__version__}")

print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('has_unsloth', HAS_UNSLOTH)
print('cuda_available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda_device', torch.cuda.get_device_name(0))



In [ ]:
# Paths + run config (auto-detect first, env override optional)

def _first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    search_order = [start, *start.parents]

    for base in search_order:
        prompt_yaml = base / 'configs' / 'prompts' / 'translation_dynamic_5.yaml'
        prompt_json = base / 'configs' / 'prompts' / 'translation_dynamic_5.json'
        sft_yaml = base / 'configs' / 'preprocessing' / 'sft_translation.yaml'
        if (prompt_yaml.exists() or prompt_json.exists()) and sft_yaml.exists():
            return base

    extra_candidates = [
        Path('/workspace/scp_stage3_it'),
        Path('/workspace'),
        start,
    ]
    for base in extra_candidates:
        prompt_yaml = base / 'configs' / 'prompts' / 'translation_dynamic_5.yaml'
        prompt_json = base / 'configs' / 'prompts' / 'translation_dynamic_5.json'
        sft_yaml = base / 'configs' / 'preprocessing' / 'sft_translation.yaml'
        if (prompt_yaml.exists() or prompt_json.exists()) and sft_yaml.exists():
            return base.resolve()

    return start


_user_work_root = os.environ.get('WORK_ROOT', '').strip()
if _user_work_root:
    WORK_ROOT = Path(_user_work_root).expanduser().resolve()
else:
    WORK_ROOT = _find_repo_root(Path.cwd())

# INPUT_CSV
_user_input_csv = os.environ.get('INPUT_CSV', '').strip()
if _user_input_csv:
    INPUT_CSV = Path(_user_input_csv).expanduser().resolve()
else:
    input_candidates = [
        WORK_ROOT / 'data' / 'eval' / 'test.csv',
        WORK_ROOT / 'data' / 'test.csv',
        WORK_ROOT / 'test.csv',
        Path('/workspace/data/eval/test.csv'),
        Path('/workspace/data/test.csv'),
        Path('/workspace/test.csv'),
    ]
    INPUT_CSV = _first_existing(input_candidates)

    if INPUT_CSV is None:
        # Last-resort local scan (exclude common cache/output dirs)
        scan_hits = []
        for p in WORK_ROOT.rglob('test.csv'):
            parts = set(p.parts)
            if '.venv' in parts or '.git' in parts or 'outputs' in parts or 'artifacts' in parts:
                continue
            scan_hits.append(p)
        scan_hits = sorted(scan_hits, key=lambda x: (len(x.parts), str(x)))
        if scan_hits:
            INPUT_CSV = scan_hits[0].resolve()

if INPUT_CSV is None:
    raise FileNotFoundError(
        'Unable to auto-detect INPUT_CSV. Set INPUT_CSV env var or place test.csv under data/eval.'
    )

# OUTPUT_DIR
_user_output_dir = os.environ.get('OUTPUT_DIR', '').strip()
if _user_output_dir:
    OUTPUT_DIR = Path(_user_output_dir).expanduser().resolve()
else:
    OUTPUT_DIR = (WORK_ROOT / 'outputs' / 'testcsv_3model').resolve()

# HF cache
_user_hf_cache = os.environ.get('HF_CACHE_ROOT', '').strip()
if _user_hf_cache:
    HF_CACHE_ROOT = Path(_user_hf_cache).expanduser().resolve()
else:
    HF_CACHE_ROOT = (OUTPUT_DIR / 'hf_cache').resolve()

# Prompt config
_user_prompt_cfg = os.environ.get('PROMPT_CONFIG_PATH', '').strip()
if _user_prompt_cfg:
    PROMPT_CONFIG_PATH = Path(_user_prompt_cfg).expanduser().resolve()
else:
    prompt_candidates = [
        WORK_ROOT / 'configs' / 'prompts' / 'translation_dynamic_5.yaml',
        WORK_ROOT / 'configs' / 'prompts' / 'translation_dynamic_5.json',
    ]
    if (WORK_ROOT / 'configs' / 'prompts').exists():
        prompt_candidates.extend(sorted((WORK_ROOT / 'configs' / 'prompts').glob('translation_dynamic_5.*')))
        prompt_candidates.extend(sorted((WORK_ROOT / 'configs' / 'prompts').glob('*.yaml')))
        prompt_candidates.extend(sorted((WORK_ROOT / 'configs' / 'prompts').glob('*.json')))
    PROMPT_CONFIG_PATH = _first_existing(prompt_candidates)

if PROMPT_CONFIG_PATH is None:
    raise FileNotFoundError(
        'Unable to auto-detect PROMPT_CONFIG_PATH. Set env var or provide configs/prompts/translation_dynamic_5.yaml(.json).'
    )

# SFT config
_user_sft_cfg = os.environ.get('SFT_CONFIG_PATH', '').strip()
if _user_sft_cfg:
    SFT_CONFIG_PATH = Path(_user_sft_cfg).expanduser().resolve()
else:
    sft_candidates = [
        WORK_ROOT / 'configs' / 'preprocessing' / 'sft_translation.yaml',
        WORK_ROOT / 'configs' / 'preprocessing' / 'sft_translation.yml',
        WORK_ROOT / 'configs' / 'preprocessing' / 'sft_translation.json',
    ]
    SFT_CONFIG_PATH = _first_existing(sft_candidates)

if SFT_CONFIG_PATH is None:
    raise FileNotFoundError(
        'Unable to auto-detect SFT_CONFIG_PATH. Set env var or provide configs/preprocessing/sft_translation.yaml.'
    )

SOURCE_LANG_ISO = os.environ.get('SOURCE_LANG_ISO', 'en').strip().lower()
TARGET_LANG_ISO = os.environ.get('TARGET_LANG_ISO', 'ko').strip().lower()
TEMPLATE_SEED = int(os.environ.get('TEMPLATE_SEED', '42'))
_fixed_template_index_env = os.environ.get('FIXED_TEMPLATE_INDEX', '').strip()
FIXED_TEMPLATE_INDEX = int(_fixed_template_index_env) if _fixed_template_index_env else None

MODELS = [
    'alwaysgood/qwen3-it',
    'alwaysgood/qwen35-it',
    'alwaysgood/gemma4-it',
]

MAX_NEW_TOKENS = int(os.environ.get('MAX_NEW_TOKENS', '512'))
DO_SAMPLE = False
TEMPERATURE = 0.0
BATCH_SIZE_QWEN = int(os.environ.get('BATCH_SIZE_QWEN', '8'))
# Keep Gemma slightly lower by default because it tends to consume more VRAM per batch
# in this notebook setup (model/runtime differences). Override via env if your GPU allows more.
BATCH_SIZE_GEMMA = int(os.environ.get('BATCH_SIZE_GEMMA', '6'))
SORT_BY_INPUT_LENGTH = True
USE_UNSLOTH = True
UNSLOTH_MAX_SEQ_LENGTH = int(os.environ.get('UNSLOTH_MAX_SEQ_LENGTH', '4096'))

# Metrics aligned with translation_eval.ipynb
METRICS = ['BLEU', 'chrF', 'TER', 'ROUGE-L', 'BERTScore-F1', 'COMET-ref']

# COMET is mandatory for this notebook.
RUN_COMET = True
COMET_REQUIRED = True
COMET_MODEL_CANDIDATES = [
    'Unbabel/XCOMET-XXL',
    'Unbabel/XCOMET-XL',
]
COMET_BATCH_SIZE = int(os.environ.get('COMET_BATCH_SIZE', '8'))
COMET_NUM_WORKERS = int(os.environ.get('COMET_NUM_WORKERS', '2'))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print('WORK_ROOT', WORK_ROOT)
print('INPUT_CSV', INPUT_CSV)
print('OUTPUT_DIR', OUTPUT_DIR)
print('HF_CACHE_ROOT', HF_CACHE_ROOT)
print('PROMPT_CONFIG_PATH', PROMPT_CONFIG_PATH)
print('SFT_CONFIG_PATH', SFT_CONFIG_PATH)
print('SOURCE_LANG_ISO', SOURCE_LANG_ISO)
print('TARGET_LANG_ISO', TARGET_LANG_ISO)
print('TEMPLATE_SEED', TEMPLATE_SEED)
print('FIXED_TEMPLATE_INDEX', FIXED_TEMPLATE_INDEX)
print('MODELS', MODELS)
print('MAX_NEW_TOKENS (generated tokens only; prompt excluded)', MAX_NEW_TOKENS)
print('USE_UNSLOTH', USE_UNSLOTH)
print('UNSLOTH_MAX_SEQ_LENGTH', UNSLOTH_MAX_SEQ_LENGTH)
print('METRICS', METRICS)
print('RUN_COMET', RUN_COMET)
print('COMET_REQUIRED', COMET_REQUIRED)
print('COMET_MODEL_CANDIDATES', COMET_MODEL_CANDIDATES)
print('COMET_BATCH_SIZE', COMET_BATCH_SIZE)





In [ ]:
# Load test.csv
if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Input CSV not found: {INPUT_CSV}')

raw_df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig').copy()

def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f'Missing required column. candidates={candidates}, available={list(df.columns)}')
    return None

id_col = pick_col(raw_df, ['ID', '\ufeffID', 'id', 'row_id'])
source_col = pick_col(raw_df, ['Source_En', 'source_text', 'source_en'])
target_col = pick_col(raw_df, ['Target_Ko', 'target_text', 'reference_text', 'target_ko'])
src_lang_col = pick_col(raw_df, ['source_lang_iso', 'src_lang_iso', 'source_lang', 'Source_Lang'], required=False)
tgt_lang_col = pick_col(raw_df, ['target_lang_iso', 'tgt_lang_iso', 'target_lang', 'Target_Lang'], required=False)

eval_df = raw_df[[id_col, source_col, target_col]].copy()
eval_df = eval_df.rename(columns={id_col: 'row_id', source_col: 'source_text', target_col: 'reference_text'})

eval_df['row_id'] = pd.to_numeric(eval_df['row_id'], errors='coerce')
eval_df = eval_df.dropna(subset=['row_id']).copy()
eval_df['row_id'] = eval_df['row_id'].astype(int)

eval_df['source_text'] = eval_df['source_text'].fillna('').astype(str)
eval_df['reference_text'] = eval_df['reference_text'].fillna('').astype(str)

if src_lang_col is not None:
    eval_df['source_lang_iso'] = raw_df.loc[eval_df.index, src_lang_col].fillna(SOURCE_LANG_ISO).astype(str).str.strip().str.lower()
else:
    eval_df['source_lang_iso'] = SOURCE_LANG_ISO

if tgt_lang_col is not None:
    eval_df['target_lang_iso'] = raw_df.loc[eval_df.index, tgt_lang_col].fillna(TARGET_LANG_ISO).astype(str).str.strip().str.lower()
else:
    eval_df['target_lang_iso'] = TARGET_LANG_ISO

eval_df['source_lang_iso'] = eval_df['source_lang_iso'].replace('', SOURCE_LANG_ISO)
eval_df['target_lang_iso'] = eval_df['target_lang_iso'].replace('', TARGET_LANG_ISO)
eval_df = eval_df.sort_values('row_id').reset_index(drop=True)

print('rows', len(eval_df))
display(eval_df.head(3))



In [ ]:
import hashlib
import json as pyjson

DEFAULT_LANG_NAME_MAP = {
    'en': 'English',
    'ko': 'Korean',
    'ja': 'Japanese',
    'zh': 'Chinese',
}

DEFAULT_LANG_LOCALE_MAP = {
    'en': 'en-US',
    'ko': 'ko-KR',
    'ja': 'ja-JP',
    'zh': 'zh-CN',
}


def slugify_model(model_id: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', model_id)


def resolve_batch_size(model_id: str) -> int:
    if 'gemma' in model_id.lower():
        return int(BATCH_SIZE_GEMMA)
    return int(BATCH_SIZE_QWEN)


def _load_structured_file(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f'Config file not found: {path}')

    suffix = path.suffix.lower()
    text = path.read_text(encoding='utf-8')

    if suffix == '.json':
        data = pyjson.loads(text)
        if not isinstance(data, dict):
            raise ValueError(f'JSON config must be an object: {path}')
        return data

    if suffix in {'.yaml', '.yml'}:
        try:
            import yaml
        except Exception as e:
            raise RuntimeError('PyYAML is required to read .yaml prompt config. Install pyyaml.') from e
        data = yaml.safe_load(text)
        if not isinstance(data, dict):
            raise ValueError(f'YAML config must be a mapping: {path}')
        return data

    raise ValueError(f'Unsupported config format: {path} (expected .json/.yaml/.yml)')


def _as_dict(maybe_dict):
    return maybe_dict if isinstance(maybe_dict, dict) else {}


def load_prompt_assets(prompt_config_path: Path, sft_config_path: Path):
    prompt_cfg = _load_structured_file(prompt_config_path)

    templates = prompt_cfg.get('templates')
    if not isinstance(templates, list) or not templates:
        prompts_block = _as_dict(prompt_cfg.get('prompts'))
        templates = prompts_block.get('templates')

    if not isinstance(templates, list) or not templates:
        raise ValueError(
            f"Prompt config must include non-empty 'templates' or 'prompts.templates': {prompt_config_path}"
        )
    prompt_templates = [str(t) for t in templates]

    sft_raw = _load_structured_file(sft_config_path)
    sft_cfg = _as_dict(sft_raw.get('sft'))
    if not sft_cfg:
        preprocessing = _as_dict(sft_raw.get('preprocessing'))
        sft_cfg = _as_dict(preprocessing.get('sft'))

    response_prefix = str(sft_cfg.get('response_prefix', 'response: '))

    lang_name_map = dict(DEFAULT_LANG_NAME_MAP)
    for k, v in _as_dict(sft_cfg.get('lang_name_map')).items():
        lang_name_map[str(k).strip().lower()] = str(v)

    lang_locale_map = dict(DEFAULT_LANG_LOCALE_MAP)
    for k, v in _as_dict(sft_cfg.get('lang_locale_map')).items():
        lang_locale_map[str(k).strip().lower()] = str(v)

    return prompt_templates, response_prefix, lang_name_map, lang_locale_map


def stable_template_index(sample_key: str, template_count: int, seed: int) -> int:
    if template_count <= 0:
        raise ValueError('template_count must be positive')
    key = f'{seed}|{sample_key}'.encode('utf-8')
    digest = hashlib.blake2b(key, digest_size=8).digest()
    value = int.from_bytes(digest, byteorder='big', signed=False)
    return value % template_count


def resolve_template_for_row(prompt_templates, row_id: str, template_seed: int, fixed_template_index: int | None):
    if not prompt_templates:
        raise ValueError('prompt_templates must not be empty')

    if fixed_template_index is not None:
        if fixed_template_index < 0 or fixed_template_index >= len(prompt_templates):
            raise ValueError(
                f'template_index={fixed_template_index} is out of range for {len(prompt_templates)} templates'
            )
        return prompt_templates[fixed_template_index], fixed_template_index

    idx = stable_template_index(sample_key=str(row_id), template_count=len(prompt_templates), seed=int(template_seed))
    return prompt_templates[idx], idx


def render_prompt(source_text: str, prompt_template: str, response_prefix: str, src_lang_name: str, tgt_lang_name: str, src_locale: str, tgt_locale: str) -> str:
    prompt = prompt_template.format(
        src_lang_name=src_lang_name,
        tgt_lang_name=tgt_lang_name,
        src_locale=src_locale,
        tgt_locale=tgt_locale,
        src=source_text,
    )
    return f'{prompt}\n{response_prefix}'


def resolve_lang_context(source_lang_iso: str, target_lang_iso: str, lang_name_map: dict, lang_locale_map: dict):
    src_iso = str(source_lang_iso or SOURCE_LANG_ISO).strip().lower()
    tgt_iso = str(target_lang_iso or TARGET_LANG_ISO).strip().lower()
    if not src_iso:
        src_iso = SOURCE_LANG_ISO
    if not tgt_iso:
        tgt_iso = TARGET_LANG_ISO

    src_lang_name = lang_name_map.get(src_iso, src_iso.upper())
    tgt_lang_name = lang_name_map.get(tgt_iso, tgt_iso.upper())
    src_locale = lang_locale_map.get(src_iso, src_iso)
    tgt_locale = lang_locale_map.get(tgt_iso, tgt_iso)
    return src_iso, tgt_iso, src_lang_name, tgt_lang_name, src_locale, tgt_locale


def clean_hypothesis(text: str, response_prefix: str) -> str:
    out = str(text or '').strip()
    normalized_prefix = str(response_prefix or '').strip()
    if normalized_prefix and normalized_prefix in out:
        out = out.rsplit(normalized_prefix, 1)[-1].strip()

    for prefix in (normalized_prefix, '<KO>', '<ko>', '[KO]', 'KO:', '번역:', 'Translation:'):
        if prefix and out.startswith(prefix):
            out = out[len(prefix):].strip()
    return out


def _load_with_unsloth(model_id: str):
    from unsloth import FastLanguageModel, FastVisionModel

    errors = []
    for backend, model_class in (("language", FastLanguageModel), ("vision", FastVisionModel)):
        try:
            unsloth_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else None
            model, tokenizer = model_class.from_pretrained(
                model_name=model_id,
                max_seq_length=int(UNSLOTH_MAX_SEQ_LENGTH),
                dtype=unsloth_dtype,
                load_in_4bit=False,
            )
            return model, tokenizer, backend, unsloth_dtype
        except Exception as e:
            errors.append(f"{backend}: {type(e).__name__}: {e}")

    joined = "\n".join(errors)
    raise RuntimeError(f"Unsloth load failed for {model_id}\n{joined}")


def load_model_and_tokenizer(model_id: str, cache_dir: Path):
    if not USE_UNSLOTH:
        raise RuntimeError('USE_UNSLOTH=False is not supported in this notebook. Set USE_UNSLOTH=True.')
    if importlib.util.find_spec('unsloth') is None:
        raise RuntimeError('unsloth is required but not installed.')

    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME'] = str(cache_dir)
    os.environ['TRANSFORMERS_CACHE'] = str(cache_dir)
    os.environ['HUGGINGFACE_HUB_CACHE'] = str(cache_dir)

    _ = AutoConfig.from_pretrained(model_id, trust_remote_code=True, cache_dir=str(cache_dir))
    model, tokenizer, backend, unsloth_dtype = _load_with_unsloth(model_id)
    print(f'  loader=unsloth/{backend} dtype={unsloth_dtype}')

    if backend == 'language':
        try:
            from unsloth import FastLanguageModel
            FastLanguageModel.for_inference(model)
            print('  unsloth_for_inference=True')
        except Exception as e:
            print(f'[WARN] unsloth for_inference failed: {type(e).__name__}: {e}')

    base_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)
    if base_tokenizer.pad_token_id is None:
        base_tokenizer.pad_token = base_tokenizer.eos_token
    base_tokenizer.padding_side = 'left'

    if hasattr(model, 'config'):
        model.config.use_cache = True
    if hasattr(model, 'generation_config'):
        model.generation_config.use_cache = True

    model.eval()
    device = next(model.parameters()).device
    return model, base_tokenizer, device


def _build_gen_kwargs(tokenizer):
    kwargs = {
        'max_new_tokens': int(MAX_NEW_TOKENS),
        'do_sample': bool(DO_SAMPLE),
        'pad_token_id': tokenizer.pad_token_id,
        'eos_token_id': tokenizer.eos_token_id,
        'use_cache': True,
        'num_beams': 1,
    }
    if DO_SAMPLE:
        kwargs['temperature'] = float(TEMPERATURE)
    return kwargs


def _generate_single(model, tokenizer, device, row: dict, context_limit: int | None, response_prefix: str):
    try:
        prompt = row['prompt']
        encoded = tokenizer(prompt, return_tensors='pt', add_special_tokens=False)
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)

        effective_max_new = int(MAX_NEW_TOKENS)
        if context_limit is not None:
            prompt_len = int(attention_mask.sum().item())
            allowed = context_limit - prompt_len
            if allowed <= 0:
                raise RuntimeError(f'input exceeds context limit: input={prompt_len} limit={context_limit}')
            effective_max_new = min(effective_max_new, allowed)

        gen_kwargs = _build_gen_kwargs(tokenizer)
        gen_kwargs['max_new_tokens'] = int(effective_max_new)

        with torch.inference_mode():
            out = model.generate(input_ids=input_ids, attention_mask=attention_mask, **gen_kwargs)

        prompt_len = int(attention_mask.sum().item())
        gen_ids = out[0][prompt_len:]
        hyp = clean_hypothesis(tokenizer.decode(gen_ids, skip_special_tokens=True), response_prefix=response_prefix)

        return {
            'row_id': int(row['row_id']),
            'source_text': row['source_text'],
            'reference_text': row['reference_text'],
            'source_lang_iso': row['source_lang_iso'],
            'target_lang_iso': row['target_lang_iso'],
            'template_index': str(row['template_index']),
            'hypothesis': hyp,
            'error': '',
        }
    except Exception as e:
        return {
            'row_id': int(row['row_id']),
            'source_text': row['source_text'],
            'reference_text': row['reference_text'],
            'source_lang_iso': row['source_lang_iso'],
            'target_lang_iso': row['target_lang_iso'],
            'template_index': str(row['template_index']),
            'hypothesis': '',
            'error': f'{type(e).__name__}: {e}',
        }


def _get_context_limit(model, tokenizer):
    model_limit = getattr(getattr(model, 'config', None), 'max_position_embeddings', None)
    if isinstance(model_limit, int) and model_limit > 0:
        return model_limit

    tokenizer_limit = getattr(tokenizer, 'model_max_length', None)
    if isinstance(tokenizer_limit, int) and 0 < tokenizer_limit < 1_000_000:
        return tokenizer_limit
    return None


def generate_for_rows(model, tokenizer, device, rows_df: pd.DataFrame, model_id: str, prompt_templates, response_prefix: str, lang_name_map: dict, lang_locale_map: dict):
    batch_size = max(1, resolve_batch_size(model_id))

    work_df = rows_df.reset_index(drop=True).copy()
    work_df['_orig_pos'] = work_df.index.astype(int)

    rendered_rows = []
    for _, row in work_df.iterrows():
        src_iso, tgt_iso, src_lang_name, tgt_lang_name, src_locale, tgt_locale = resolve_lang_context(
            source_lang_iso=row.get('source_lang_iso', SOURCE_LANG_ISO),
            target_lang_iso=row.get('target_lang_iso', TARGET_LANG_ISO),
            lang_name_map=lang_name_map,
            lang_locale_map=lang_locale_map,
        )
        tmpl, tmpl_idx = resolve_template_for_row(
            prompt_templates=prompt_templates,
            row_id=str(int(row['row_id'])),
            template_seed=int(TEMPLATE_SEED),
            fixed_template_index=FIXED_TEMPLATE_INDEX,
        )
        prompt = render_prompt(
            source_text=str(row['source_text']),
            prompt_template=tmpl,
            response_prefix=response_prefix,
            src_lang_name=src_lang_name,
            tgt_lang_name=tgt_lang_name,
            src_locale=src_locale,
            tgt_locale=tgt_locale,
        )

        rendered_rows.append({
            '_orig_pos': int(row['_orig_pos']),
            'row_id': int(row['row_id']),
            'source_text': str(row['source_text']),
            'reference_text': str(row['reference_text']),
            'source_lang_iso': src_iso,
            'target_lang_iso': tgt_iso,
            'template_index': int(tmpl_idx),
            'prompt': prompt,
        })

    if SORT_BY_INPUT_LENGTH:
        rendered_rows.sort(key=lambda r: (len(r['source_text']), r['_orig_pos']))

    results = []
    total = len(rendered_rows)
    context_limit = _get_context_limit(model, tokenizer)
    print(f'infer_batch_size={batch_size}, sort_by_length={SORT_BY_INPUT_LENGTH}, context_limit={context_limit}')

    for start in range(0, total, batch_size):
        chunk = rendered_rows[start:start + batch_size]
        prompts = [r['prompt'] for r in chunk]

        try:
            encoded = tokenizer(
                prompts,
                return_tensors='pt',
                add_special_tokens=False,
                padding=True,
                truncation=False,
                pad_to_multiple_of=8,
            )
            input_ids = encoded['input_ids'].to(device)
            attention_mask = encoded['attention_mask'].to(device)
            batch_input_length = int(input_ids.shape[1])

            effective_max_new = int(MAX_NEW_TOKENS)
            if context_limit is not None:
                prompt_lens = attention_mask.sum(dim=1)
                allowed = int((context_limit - prompt_lens).min().item())
                if allowed <= 0:
                    raise RuntimeError(
                        f'input exceeds context limit in batch: max_prompt={int(prompt_lens.max().item())} limit={context_limit}'
                    )
                effective_max_new = min(effective_max_new, allowed)

            gen_kwargs = _build_gen_kwargs(tokenizer)
            gen_kwargs['max_new_tokens'] = int(effective_max_new)

            with torch.inference_mode():
                out = model.generate(input_ids=input_ids, attention_mask=attention_mask, **gen_kwargs)

            for i, row in enumerate(chunk):
                gen_ids = out[i][batch_input_length:]
                hyp = clean_hypothesis(tokenizer.decode(gen_ids, skip_special_tokens=True), response_prefix=response_prefix)
                results.append({
                    '_orig_pos': row['_orig_pos'],
                    'row_id': int(row['row_id']),
                    'source_text': row['source_text'],
                    'reference_text': row['reference_text'],
                    'source_lang_iso': row['source_lang_iso'],
                    'target_lang_iso': row['target_lang_iso'],
                    'template_index': str(row['template_index']),
                    'hypothesis': hyp,
                    'error': '',
                })

        except RuntimeError as e:
            if 'out of memory' in str(e).lower() and len(chunk) > 1:
                print(f'[OOM] chunk {start}:{start + len(chunk)} -> fallback single')
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                for row in chunk:
                    one = _generate_single(model, tokenizer, device, row=row, context_limit=context_limit, response_prefix=response_prefix)
                    one['_orig_pos'] = row['_orig_pos']
                    results.append(one)
            else:
                err = f'{type(e).__name__}: {e}'
                for row in chunk:
                    results.append({
                        '_orig_pos': row['_orig_pos'],
                        'row_id': int(row['row_id']),
                        'source_text': row['source_text'],
                        'reference_text': row['reference_text'],
                        'source_lang_iso': row['source_lang_iso'],
                        'target_lang_iso': row['target_lang_iso'],
                        'template_index': str(row['template_index']),
                        'hypothesis': '',
                        'error': err,
                    })

        except Exception as e:
            err = f'{type(e).__name__}: {e}'
            for row in chunk:
                results.append({
                    '_orig_pos': row['_orig_pos'],
                    'row_id': int(row['row_id']),
                    'source_text': row['source_text'],
                    'reference_text': row['reference_text'],
                    'source_lang_iso': row['source_lang_iso'],
                    'target_lang_iso': row['target_lang_iso'],
                    'template_index': str(row['template_index']),
                    'hypothesis': '',
                    'error': err,
                })

        done = min(start + len(chunk), total)
        if done % 20 == 0 or done == total:
            print(f'progress {done}/{total}')

    results.sort(key=lambda x: x.get('_orig_pos', 0))
    for row in results:
        row.pop('_orig_pos', None)
    return results


def cleanup_model_cache(cache_dir: Path, model=None, tokenizer=None):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    if cache_dir.exists():
        shutil.rmtree(cache_dir, ignore_errors=True)


PROMPT_TEMPLATES, RESPONSE_PREFIX, LANG_NAME_MAP, LANG_LOCALE_MAP = load_prompt_assets(
    prompt_config_path=PROMPT_CONFIG_PATH,
    sft_config_path=SFT_CONFIG_PATH,
)

if FIXED_TEMPLATE_INDEX is not None:
    if FIXED_TEMPLATE_INDEX < 0 or FIXED_TEMPLATE_INDEX >= len(PROMPT_TEMPLATES):
        raise ValueError(
            f'FIXED_TEMPLATE_INDEX={FIXED_TEMPLATE_INDEX} out of range for {len(PROMPT_TEMPLATES)} templates'
        )

print('prompt_templates_count', len(PROMPT_TEMPLATES))
print('response_prefix', repr(RESPONSE_PREFIX))






In [ ]:
# Sequential generation (model-by-model)
all_result_frames = []
expected_rows = len(eval_df)
expected_row_ids = set(eval_df['row_id'].astype(int).tolist())

for model_id in MODELS:
    slug = slugify_model(model_id)
    cache_dir = HF_CACHE_ROOT / slug
    per_model_csv = OUTPUT_DIR / f'{slug}__translations.csv'

    print('\n' + '=' * 100)
    print('[MODEL]', model_id)
    print('=' * 100)

    if per_model_csv.exists():
        existing = pd.read_csv(per_model_csv, encoding='utf-8-sig')
        required_cols = [
            'row_id', 'model_name', 'source_text', 'reference_text',
            'source_lang_iso', 'target_lang_iso', 'template_index', 'hypothesis', 'error',
        ]
        valid_existing = False

        if set(required_cols).issubset(set(existing.columns)) and len(existing) == expected_rows:
            existing = existing[required_cols].copy()
            existing['row_id'] = pd.to_numeric(existing['row_id'], errors='coerce')
            existing = existing.dropna(subset=['row_id']).copy()
            existing['row_id'] = existing['row_id'].astype(int)

            if len(existing) == expected_rows:
                row_id_ok = set(existing['row_id'].tolist()) == expected_row_ids
                row_id_unique_ok = existing['row_id'].nunique() == expected_rows
                model_name_values = set(existing['model_name'].astype(str).dropna().unique().tolist())
                model_name_ok = model_name_values == {model_id}
                valid_existing = row_id_ok and row_id_unique_ok and model_name_ok

        if valid_existing:
            print('skip existing (validated):', per_model_csv)
            all_result_frames.append(existing[required_cols].copy())
            continue

        print('existing output invalid; regenerating:', per_model_csv)

    model = tokenizer = None
    try:
        model, tokenizer, device = load_model_and_tokenizer(model_id, cache_dir)
        rows = generate_for_rows(
            model,
            tokenizer,
            device,
            eval_df,
            model_id,
            prompt_templates=PROMPT_TEMPLATES,
            response_prefix=RESPONSE_PREFIX,
            lang_name_map=LANG_NAME_MAP,
            lang_locale_map=LANG_LOCALE_MAP,
        )
        per_df = pd.DataFrame(rows)
        per_df['model_name'] = model_id
        per_df = per_df[[
            'row_id', 'model_name', 'source_text', 'reference_text',
            'source_lang_iso', 'target_lang_iso', 'template_index', 'hypothesis', 'error',
        ]]
        per_df.to_csv(per_model_csv, index=False, encoding='utf-8-sig')
        print('saved:', per_model_csv)
        all_result_frames.append(per_df.copy())
    finally:
        cleanup_model_cache(cache_dir, model=model, tokenizer=tokenizer)

print('\nAll model runs completed.')



In [ ]:
# Merge outputs
translations_path = OUTPUT_DIR / 'translations.csv'

if not all_result_frames:
    raise RuntimeError('No results collected from generation step.')

translations_df = pd.concat(all_result_frames, axis=0, ignore_index=True)
translations_df = translations_df[[
    'row_id', 'model_name', 'source_text', 'reference_text',
    'source_lang_iso', 'target_lang_iso', 'template_index', 'hypothesis', 'error',
]]
translations_df = translations_df.sort_values(['row_id', 'model_name']).reset_index(drop=True)
translations_df.to_csv(translations_path, index=False, encoding='utf-8-sig')

print('saved:', translations_path)
print('total_rows:', len(translations_df))
print('rows_per_model:')
display(translations_df.groupby('model_name')['row_id'].count().sort_values(ascending=False))
print('error_rows_per_model:')
display(translations_df.assign(has_error=translations_df['error'].fillna('').str.strip().ne('')).groupby('model_name')['has_error'].sum())
display(translations_df.head(3))

# Wide table for easy side-by-side comparison per row
wide_df = (
    translations_df
    .pivot_table(
        index=['row_id', 'source_text', 'reference_text'],
        columns='model_name',
        values='hypothesis',
        aggfunc='first',
    )
    .reset_index()
)
wide_df.columns.name = None
wide_df = wide_df.sort_values('row_id').reset_index(drop=True)
wide_path = OUTPUT_DIR / 'translations_by_model_wide.csv'
wide_df.to_csv(wide_path, index=False, encoding='utf-8-sig')

print('saved:', wide_path)
print('wide_rows:', len(wide_df))
display(wide_df.head(5))



In [ ]:
# Row-level metrics
score_rows = []

valid_df = translations_df[
    translations_df['error'].fillna('').eq('')
    & translations_df['hypothesis'].fillna('').astype(str).str.strip().ne('')
    & translations_df['reference_text'].fillna('').astype(str).str.strip().ne('')
].copy()

print('valid rows for metric:', len(valid_df))

if valid_df.empty:
    raise ValueError('No valid rows for metric computation.')

if any(m in METRICS for m in ['BLEU', 'chrF', 'TER']):
    from sacrebleu.metrics import BLEU, CHRF, TER
    bleu_metric = BLEU(effective_order=True, smooth_method='exp')
    chrf_metric = CHRF(word_order=2)
    ter_metric = TER()

    if 'BLEU' in METRICS:
        for _, r in tqdm(valid_df.iterrows(), total=len(valid_df), desc='BLEU'):
            score_rows.append({
                'row_id': int(r['row_id']),
                'model_name': str(r['model_name']),
                'metric_name': 'BLEU',
                'metric_value': float(bleu_metric.sentence_score(str(r['hypothesis']), [str(r['reference_text'])]).score),
            })

    if 'chrF' in METRICS:
        for _, r in tqdm(valid_df.iterrows(), total=len(valid_df), desc='chrF'):
            score_rows.append({
                'row_id': int(r['row_id']),
                'model_name': str(r['model_name']),
                'metric_name': 'chrF',
                'metric_value': float(chrf_metric.sentence_score(str(r['hypothesis']), [str(r['reference_text'])]).score),
            })

    if 'TER' in METRICS:
        for _, r in tqdm(valid_df.iterrows(), total=len(valid_df), desc='TER'):
            score_rows.append({
                'row_id': int(r['row_id']),
                'model_name': str(r['model_name']),
                'metric_name': 'TER',
                'metric_value': float(ter_metric.sentence_score(str(r['hypothesis']), [str(r['reference_text'])]).score),
            })

if 'ROUGE-L' in METRICS:
    try:
        from rouge_score import rouge_scorer
        rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
        for _, r in tqdm(valid_df.iterrows(), total=len(valid_df), desc='ROUGE-L'):
            score = rouge.score(str(r['reference_text']), str(r['hypothesis']))['rougeL'].fmeasure
            score_rows.append({
                'row_id': int(r['row_id']),
                'model_name': str(r['model_name']),
                'metric_name': 'ROUGE-L',
                'metric_value': float(score),
            })
    except Exception as e:
        print('[WARN] ROUGE-L skipped:', e)

if 'BERTScore-F1' in METRICS:
    try:
        from bert_score import score as bertscore_score
        for model_name, g in tqdm(valid_df.groupby('model_name'), total=valid_df['model_name'].nunique(), desc='BERTScore'):
            cands = g['hypothesis'].astype(str).tolist()
            refs = g['reference_text'].astype(str).tolist()
            _, _, f1 = bertscore_score(cands, refs, lang='ko', rescale_with_baseline=False, verbose=False)
            for rid, val in zip(g['row_id'].astype(int).tolist(), f1.tolist()):
                score_rows.append({
                    'row_id': int(rid),
                    'model_name': str(model_name),
                    'metric_name': 'BERTScore-F1',
                    'metric_value': float(val),
                })
    except Exception as e:
        print('[WARN] BERTScore-F1 skipped:', e)

if 'COMET-ref' in METRICS:
    if COMET_REQUIRED and not RUN_COMET:
        raise RuntimeError('COMET-ref is required, but RUN_COMET=False. Set RUN_COMET=True.')

    if RUN_COMET:
        from comet import download_model, load_from_checkpoint

        selected_comet_model = None
        comet_model = None
        last_err = None
        for candidate in COMET_MODEL_CANDIDATES:
            try:
                print(f'[COMET] trying model: {candidate}')
                comet_model_path = download_model(candidate)
                comet_model = load_from_checkpoint(comet_model_path)
                selected_comet_model = candidate
                break
            except Exception as e:
                last_err = e
                print(f'[COMET][WARN] failed to load {candidate}: {type(e).__name__}: {e}')

        if comet_model is None:
            raise RuntimeError(f'Unable to load any COMET model from {COMET_MODEL_CANDIDATES}') from last_err

        print(f'[COMET] selected model: {selected_comet_model}')
        gpus = 1 if torch.cuda.is_available() else 0

        for model_name, g in tqdm(valid_df.groupby('model_name'), total=valid_df['model_name'].nunique(), desc='COMET-ref'):
            data = [
                {'src': s, 'mt': h, 'ref': r}
                for s, h, r in zip(g['source_text'].astype(str), g['hypothesis'].astype(str), g['reference_text'].astype(str))
            ]
            pred = comet_model.predict(
                data,
                batch_size=int(COMET_BATCH_SIZE),
                gpus=gpus,
                progress_bar=False,
                num_workers=int(COMET_NUM_WORKERS),
            )
            for rid, val in zip(g['row_id'].astype(int).tolist(), pred.scores):
                score_rows.append({
                    'row_id': int(rid),
                    'model_name': str(model_name),
                    'metric_name': 'COMET-ref',
                    'metric_value': float(val),
                })

scores_df = pd.DataFrame(score_rows, columns=['row_id', 'model_name', 'metric_name', 'metric_value'])
scores_df = scores_df.sort_values(['row_id', 'model_name', 'metric_name']).reset_index(drop=True)

if 'COMET-ref' in METRICS and COMET_REQUIRED:
    expected = len(valid_df)
    actual = int((scores_df['metric_name'] == 'COMET-ref').sum())
    if actual != expected:
        raise RuntimeError(f'COMET-ref required but incomplete. expected_rows={expected}, actual_rows={actual}')

scores_path = OUTPUT_DIR / 'row_level_scores.csv'
scores_df.to_csv(scores_path, index=False, encoding='utf-8-sig')

print('saved:', scores_path)
print('score_rows:', len(scores_df))
display(scores_df.head(10))



In [ ]:
# Metric summary + composite ranking
if scores_df.empty:
    raise ValueError('No score rows. Run metric cell first.')

higher_is_better = {
    'BLEU': True,
    'chrF': True,
    'TER': False,
    'ROUGE-L': True,
    'BERTScore-F1': True,
    'COMET-ref': True,
}

summary = (
    scores_df
    .groupby(['model_name', 'metric_name'], as_index=False)
    .agg(mean=('metric_value', 'mean'), std=('metric_value', 'std'))
)

rank_rows = []
for metric_name, g in summary.groupby('metric_name'):
    asc = not higher_is_better.get(metric_name, True)
    g2 = g.sort_values('mean', ascending=asc).reset_index(drop=True)
    g2['rank'] = np.arange(1, len(g2) + 1)
    rank_rows.append(g2)
summary = pd.concat(rank_rows, axis=0, ignore_index=True)

norm_rows = []
for metric_name, g in summary.groupby('metric_name'):
    vals = g['mean'].astype(float).values
    vmin, vmax = float(np.min(vals)), float(np.max(vals))
    if vmax - vmin < 1e-12:
        norm = np.ones_like(vals)
    else:
        norm = (vals - vmin) / (vmax - vmin)
    if not higher_is_better.get(metric_name, True):
        norm = 1.0 - norm
    for model_name, n in zip(g['model_name'].tolist(), norm.tolist()):
        norm_rows.append({'model_name': model_name, 'metric_name': metric_name, 'normalized': float(n)})

norm_df = pd.DataFrame(norm_rows)
composite = norm_df.groupby('model_name', as_index=False)['normalized'].mean().rename(columns={'normalized': 'composite_score'})
composite['composite_rank'] = composite['composite_score'].rank(ascending=False, method='min').astype(int)

model_summary = summary.merge(composite, on='model_name', how='left')
model_summary = model_summary[['model_name', 'metric_name', 'mean', 'std', 'rank', 'composite_score', 'composite_rank']]
model_summary = model_summary.sort_values(['metric_name', 'rank', 'model_name']).reset_index(drop=True)
summary_path = OUTPUT_DIR / 'model_summary.csv'
model_summary.to_csv(summary_path, index=False, encoding='utf-8-sig')

print('saved:', summary_path)
display(model_summary)
display(composite.sort_values('composite_score', ascending=False))

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    plt.figure(figsize=(8, 4))
    comp_df = composite.sort_values('composite_score', ascending=False)
    sns.barplot(data=comp_df, x='model_name', y='composite_score', hue='model_name', legend=False)
    plt.xticks(rotation=20, ha='right')
    plt.title('Composite Score (higher is better)')
    plt.tight_layout()
    fig_path = OUTPUT_DIR / 'fig_composite_bar.png'
    plt.savefig(fig_path, dpi=160)
    plt.show()
    print('saved:', fig_path)
except Exception as e:
    print('[WARN] figure skipped:', e)
